# 🐍 Day 7: Mini-Project - URL Shortener

- Full-stack URL shortener
- Short code generation
- Redirect and analytics
- FastAPI + SQLite

## Project Overview

We'll build a URL shortener:
- **Create**: POST /shorten with long URL -> returns short code
- **Redirect**: GET /{code} -> 302 redirect to long URL
- **Storage**: SQLite (or in-memory for demo)
- **Short code**: Random 6-char alphanumeric

## Step 1: Database Model

Links table: id, short_code, long_url, created_at, clicks.

In [ ]:
import sqlite3
import string
import random
from datetime import datetime

def init_db():
    conn = sqlite3.connect("shortener.db")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS links (
            id INTEGER PRIMARY KEY,
            short_code TEXT UNIQUE NOT NULL,
            long_url TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            clicks INTEGER DEFAULT 0
        )
    """)
    conn.commit()
    conn.close()

init_db()
print("DB initialized")

## Step 2: Short Code Generation

Generate random 6-char code. Check uniqueness.

In [ ]:
CHARS = string.ascii_letters + string.digits

def generate_code(length=6):
    return "".join(random.choices(CHARS, k=length))

def get_unique_code(conn):
    for _ in range(10):
        code = generate_code()
        cur = conn.execute("SELECT 1 FROM links WHERE short_code = ?", (code,))
        if cur.fetchone() is None:
            return code
    raise ValueError("Could not generate unique code")

print("Sample code:", generate_code())

## Step 3: Create Link

Insert long_url, get short_code, return.

In [ ]:
def create_link(long_url: str) -> str:
    conn = sqlite3.connect("shortener.db")
    try:
        code = get_unique_code(conn)
        conn.execute(
            "INSERT INTO links (short_code, long_url) VALUES (?, ?)",
            (code, long_url)
        )
        conn.commit()
        return code
    finally:
        conn.close()

code = create_link("https://example.com/long-url")
print("Created:", code)

## Step 4: Get and Redirect

Lookup by short_code, increment clicks, return long_url.

In [ ]:
def get_link(code: str) -> str | None:
    conn = sqlite3.connect("shortener.db")
    try:
        cur = conn.execute(
            "SELECT long_url FROM links WHERE short_code = ?",
            (code,)
        )
        row = cur.fetchone()
        if row:
            conn.execute(
                "UPDATE links SET clicks = clicks + 1 WHERE short_code = ?",
                (code,)
            )
            conn.commit()
            return row[0]
        return None
    finally:
        conn.close()

url = get_link(code)
print("Resolved:", url)

## Step 5: FastAPI App

POST /shorten, GET /{code} redirect.

In [ ]:
from urllib.parse import urlparse
from fastapi import FastAPI, HTTPException
from fastapi.responses import RedirectResponse
from pydantic import BaseModel

app = FastAPI(title="URL Shortener")

class ShortenRequest(BaseModel):
    url: str

def is_valid_url(url: str) -> bool:
    try:
        result = urlparse(url)
        return all([result.scheme, result.netloc])
    except Exception:
        return False

@app.post("/shorten")
def shorten(req: ShortenRequest):
    if not is_valid_url(req.url):
        raise HTTPException(400, "Invalid URL")
    code = create_link(req.url)
    return {"short_code": code, "short_url": f"/{code}"}

@app.get("/{code}")
def redirect(code: str):
    url = get_link(code)
    if not url:
        raise HTTPException(404, "Not found")
    return RedirectResponse(url=url, status_code=302)

print("FastAPI routes: POST /shorten, GET /{code}")

## Step 6: Stats Endpoint

GET /{code}/stats returns clicks.

In [ ]:
def get_stats(code: str) -> dict | None:
    conn = sqlite3.connect("shortener.db")
    conn.row_factory = sqlite3.Row
    cur = conn.execute(
        "SELECT short_code, long_url, clicks, created_at FROM links WHERE short_code = ?",
        (code,)
    )
    row = cur.fetchone()
    conn.close()
    return dict(row) if row else None

@app.get("/{code}/stats")
def stats(code: str):
    data = get_stats(code)
    if not data:
        raise HTTPException(404, "Not found")
    return data

print("GET /{code}/stats for analytics")

## Step 7: URL Validation

The shorten route in Step 5 already validates URLs with is_valid_url() before creating. Add urlparse import at top of app.

In [ ]:
# Validation is in Step 5. is_valid_url checks scheme and netloc.
print("URL validation in shorten route")

## Complete Flow

1. POST /shorten {"url": "https://..."} -> {"short_code": "abc123"}
2. GET /abc123 -> 302 to long URL
3. GET /abc123/stats -> {"clicks": 5, ...}

## Extensions to Try

- Custom short codes (user provides)
- Expiration (TTL)
- Rate limiting
- Frontend with HTML form
- Docker + Redis for production

## Key Takeaways

- SQLite for storage, random short codes
- RedirectResponse(302) for redirect
- Validate URLs, track clicks
- Full CRUD + redirect in one app

## 🎉 Month 4 Complete!

You've completed Web & APIs: HTTP, requests, BeautifulSoup, REST, Flask, FastAPI, databases, Redis, Docker, and CI/CD. Congratulations!